# Split-Hygiene Audit (CLASSIFIER)

**Hard-fails** with `AssertionError` if any of the following is wrong:

1. **Subject-overlap** — a single `Repseudonym` is found in more than one of the train/val/test split CSVs.
2. **Duplicate matrices** — two distinct `.npz` files contain the exact same array (upstream preprocessing bug).
3. **StratifiedGroupKFold** — re-running CV with the saved seed produces overlapping validation folds.
4. **GAAE ↔ downstream cohort policy** — the chosen policy (`shared` or `disjoint`) is asserted.

Wire this notebook's checks into every production notebook via `from common.sanity import run_full_audit`.

In [1]:
import sys
from pathlib import Path
import os
import pandas as pd

V2_ROOT = Path('/mnt/e/fyassine/ad-early-detection/CLASSIFIER')
if str(V2_ROOT) not in sys.path:
    sys.path.insert(0, str(V2_ROOT))

from common.sanity import (
    assert_splits_clean,
    assert_no_duplicate_matrices,
    audit_groupkfold,
    assert_cohort_policy,
)
print('Sanity utilities loaded')

Sanity utilities loaded


## Configure paths

Edit the constants below if you ship a new data version (`__fc_dmn_sch200_flat__` → `__fc_hippo_tian2_flat__`, etc.). Everything downstream picks them up automatically.

In [2]:
DATA_VERSION   = '__fc_wholebrain_sch200_flat__'
DATA_ROOT      = Path('/mnt/e/fyassine/ad-early-detection/DATA/DELCODE') / DATA_VERSION
MATRICES_DIR   = DATA_ROOT / 'matrices'
METADATA_DIR   = DATA_ROOT / 'metadata'
COHORTS_CSV    = METADATA_DIR / 'cohorts.csv'

# Splits used by GELSTM / Long-GEC-MLP / GAAE-LogReg notebooks.
SPLITS_DIR     = METADATA_DIR / 'splits_gaae'
SPLIT_CSVS = {
    'train': str(SPLITS_DIR / 'train.csv'),
    'val':   str(SPLITS_DIR / 'val.csv'),
    'test':  str(SPLITS_DIR / 'test.csv'),
}

# GEC variant (single-scan classifier) may live in a different split dir.
SPLITS_DIR_GEC = METADATA_DIR / 'splits_gec'
SPLIT_CSVS_GEC = {
    'train': str(SPLITS_DIR_GEC / 'train.csv'),
    'val':   str(SPLITS_DIR_GEC / 'val.csv'),
    'test':  str(SPLITS_DIR_GEC / 'test.csv'),
}

for label, mapping in [('GAAE/GELSTM', SPLIT_CSVS), ('GEC', SPLIT_CSVS_GEC)]:
    for name, p in mapping.items():
        exists = os.path.exists(p)
        print(f'  [{label}] {name:<6} {p}  {"OK" if exists else "MISSING"}')

  [GAAE/GELSTM] train  /mnt/e/fyassine/ad-early-detection/DATA/DELCODE/__fc_wholebrain_sch200_flat__/metadata/splits_gaae/train.csv  OK
  [GAAE/GELSTM] val    /mnt/e/fyassine/ad-early-detection/DATA/DELCODE/__fc_wholebrain_sch200_flat__/metadata/splits_gaae/val.csv  OK
  [GAAE/GELSTM] test   /mnt/e/fyassine/ad-early-detection/DATA/DELCODE/__fc_wholebrain_sch200_flat__/metadata/splits_gaae/test.csv  OK
  [GEC] train  /mnt/e/fyassine/ad-early-detection/DATA/DELCODE/__fc_wholebrain_sch200_flat__/metadata/splits_gec/train.csv  OK
  [GEC] val    /mnt/e/fyassine/ad-early-detection/DATA/DELCODE/__fc_wholebrain_sch200_flat__/metadata/splits_gec/val.csv  OK
  [GEC] test   /mnt/e/fyassine/ad-early-detection/DATA/DELCODE/__fc_wholebrain_sch200_flat__/metadata/splits_gec/test.csv  OK


## 1. Subject-overlap audit

For every pair of split CSVs, assert that the sets of `Repseudonym` values are disjoint. Raises `AssertionError` listing the offenders on any overlap.

In [3]:
rep_gaae = assert_splits_clean(SPLIT_CSVS, id_col='Repseudonym', raise_on_overlap=True)
print('GAAE/GELSTM split sizes (subjects):', rep_gaae['sizes'])
print('Pairwise-disjoint:', rep_gaae['clean'])

GAAE/GELSTM split sizes (subjects): {'train': 345, 'val': 69, 'test': 46}
Pairwise-disjoint: True


In [4]:
if all(os.path.exists(p) for p in SPLIT_CSVS_GEC.values()):
    rep_gec = assert_splits_clean(SPLIT_CSVS_GEC, id_col='Repseudonym', raise_on_overlap=True)
    print('GEC split sizes (subjects):', rep_gec['sizes'])
    print('Pairwise-disjoint:', rep_gec['clean'])
else:
    print('GEC splits not present at expected paths — skipping')

GEC split sizes (subjects): {'train': 86, 'val': 16, 'test': 14}
Pairwise-disjoint: True


## 2. Duplicate-matrix audit

Content-hashes every `.npz` correlation matrix referenced by the production splits. If two files hash to the same value, an upstream preprocessing step produced duplicate scans — these contaminate any train/test boundary regardless of split hygiene.

In [5]:
import glob

all_npz = sorted(glob.glob(str(MATRICES_DIR / '*_z_transformed.npz')))
print(f'Found {len(all_npz)} .npz files in {MATRICES_DIR}')

dup_rep = assert_no_duplicate_matrices(all_npz, raise_on_dup=False)
print(f'  unique content hashes: {dup_rep["n_unique"]} / {dup_rep["n_files"]}')
if dup_rep['clean']:
    print('  Clean: no two files share content.')
else:
    print(f'  Found {len(dup_rep["duplicates"])} duplicate group(s):')
    for d in dup_rep['duplicates'][:5]:
        print(f'    hash={d["hash"][:10]}  paths={d["paths"]}')
    raise AssertionError('Duplicate matrices found — see list above')

Found 1186 .npz files in /mnt/e/fyassine/ad-early-detection/DATA/DELCODE/__fc_wholebrain_sch200_flat__/matrices
  unique content hashes: 1186 / 1186
  Clean: no two files share content.


## 3. StratifiedGroupKFold re-check

Rebuild the CV pool the way the production notebooks do (train + val merged), then re-run `StratifiedGroupKFold(n_splits=5, seed=42)` and assert that no subject appears in two validation folds.

In [6]:
train_df = pd.read_csv(SPLIT_CSVS['train'])
val_df   = pd.read_csv(SPLIT_CSVS['val'])
cv_pool  = pd.concat([train_df, val_df], ignore_index=True)
cv_pool  = cv_pool[cv_pool['diagnosis'].isin(['mci', 'converter'])].copy()
cv_pool['Repseudonym'] = cv_pool['Repseudonym'].astype(str)

subjects = cv_pool['Repseudonym'].tolist()
labels   = (cv_pool['diagnosis'] == 'converter').astype(int).tolist()
print(f'CV pool: {len(subjects)} subjects ({sum(labels)} converter, {len(subjects)-sum(labels)} stable)')

kf_rep = audit_groupkfold(subjects, labels, n_splits=5, seed=42)
print('Per-fold sizes:', [(f["n_train"], f["n_val"]) for f in kf_rep['folds']])
print('Pairwise-disjoint val folds:', kf_rep['clean'])

CV pool: 141 subjects (52 converter, 89 stable)
Per-fold sizes: [(113, 28), (113, 28), (113, 28), (112, 29), (113, 28)]
Pairwise-disjoint val folds: True


## 4. Cohort policy (GAAE pretrain ↔ downstream classifier)

State the policy explicitly. Two valid choices:

* `policy='shared'` — GAAE was pretrained **on the same subjects** later used downstream. Acceptable because GAAE is unsupervised, but you must say so.
* `policy='disjoint'` — GAAE pretrained on subjects never seen at classification time. Stricter, eliminates any unsupervised-representation leakage.

Whichever you pick, it goes on the methods page of the paper.

In [8]:
# Edit this to match the project's actual choice.
COHORT_POLICY = 'shared'

gaae_train = pd.read_csv(SPLIT_CSVS['train'])
gaae_subjects = gaae_train['Repseudonym'].astype(str).tolist()
downstream_subjects = subjects   # MCI/converter pool from cell above

policy_rep = assert_cohort_policy(
    gaae_pretrain_subjects=gaae_subjects,
    downstream_subjects=downstream_subjects,
    policy=COHORT_POLICY,
)
print(policy_rep)

{'policy': 'shared', 'gaae_n': 345, 'downstream_n': 141, 'shared_n': 118, 'ok': True}


## Summary

If every cell above ran without `AssertionError`, splits are clean. Run this notebook end-to-end whenever you regenerate splits or change data versions.